# P04 · Notebook 29 · A scoring that knows the two domains (DCASE 2022 Task 2, no training)

**Where we stand.** Notebook 28 measured the kept system as it is on DCASE 2022 Task 2: official score 70.63 %, second beside the 31 teams. Its development files show the main weakness: the two domains' scores sit at different levels, and the official AUC sets one domain's normal clips against both domains' anomalies.

**What it does.** It trains nothing. It reads notebook 28's saved embeddings from `P04_DomainShift` and tries three fixed scorings that treat the source and the target training clips apart (NN, DN, KM in the rules).
1. The 21 zips are downloaded again and checked, only to rebuild the list of clips (names, labels, domains).
2. **Gate G-SAME:** the system as it is, recomputed here, must give notebook 28's saved scores on every test clip. That proves the rows are the same clips.
3. **The development set** ranks six candidates (each scoring: the frozen recipe alone, and fused with the view).
4. **The choice is a rule:** the best candidate is adopted only if it beats the system as it is by at least 0.01 of the development official score.
5. **Only then** is the evaluation set scored, once, with the official evaluator. If nothing is adopted, the evaluation set is not touched.

**Rules:** `results/mimii_beats/RULES_domain_backends_2022_written_before_running.md`, written before this notebook existed.

**It needs** in Google Drive: `P04_DomainShift`, as notebook 28 left it. It writes to `P04_DomainShift/n29_backends`.

**Runtime:** any runtime (no GPU is needed; G4 is fine). Runtime › Run all. About 20 to 30 minutes, most of it the 12.4 GB download and check.

**At the end** `p04_n29_results.zip` downloads. Send it back here.

In [ ]:
# 1 · Setup
import os, sys, json, time, hashlib, struct, zipfile, subprocess, shutil, urllib.request, re, math, gc, csv, base64, importlib
from pathlib import Path
from collections import Counter, deque
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import Parameter
from sklearn.metrics import roc_auc_score
try:
    import torchaudio
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torchaudio"], check=True)
    import torchaudio

TEST = os.environ.get("P04_TEST") == "1"
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    MY = Path("/content/drive/MyDrive")
    OUT, WORK, CKPT_DIR = MY / "P04_DomainShift", Path("/content/work"), MY / "P04_BEATs"
else:
    OUT, WORK = Path(os.environ.get("P04_OUT", "P04_DomainShift")), Path(os.environ.get("P04_WORK", "work"))
    CKPT_DIR = Path(os.environ.get("P04_CKPT_DIR", "P04_BEATs"))
if not (OUT / "domain_shift.json").exists():
    raise SystemExit(f"missing {OUT}/domain_shift.json: this notebook reads the folder P04_DomainShift that notebook 28 left in Google Drive › MyDrive")
WORK.mkdir(parents=True, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
AMP = DEVICE == "cuda"
GPU = torch.cuda.get_device_name(0) if DEVICE == "cuda" else ""
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 1e9 if DEVICE == "cuda" else 0.0
T_START = time.time()
DRIVE_FREE_GB = shutil.disk_usage(OUT).free / 1e9
print("device:", DEVICE, GPU, f"({GPU_GB:.0f} GB)" if GPU else "", "· torch", torch.__version__, "· torchaudio", torchaudio.__version__,
      f"· free local disk {shutil.disk_usage(WORK).free / 1e9:.0f} GB · free Drive {DRIVE_FREE_GB:.1f} GB")


def md5(path, chunk=8 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()


def sha256(path, chunk=8 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()


def download(url, dest):
    dest = Path(dest); part = dest.with_suffix(dest.suffix + ".part")
    try:
        if url.startswith("file://"):
            shutil.copy(url[7:], part)
        elif shutil.which("wget"):
            subprocess.run(["wget", "-q", "--show-progress", "-c", "-O", str(part), url], check=True)
        else:
            urllib.request.urlretrieve(url, part)
        part.rename(dest)
    finally:
        if part.exists() and not dest.exists():
            part.unlink()


download_one = download                              # notebook 11's single-connection download


def download(url, dest):
    # several connections per file with aria2c (notebook 19 saw about 2.4 MB/s per connection from Zenodo); falls back to one connection
    dest = Path(dest); part = dest.with_suffix(dest.suffix + ".part")
    if not url.startswith("file://"):
        if shutil.which("aria2c") is None and IN_COLAB:
            if subprocess.run(["apt-get", "-qq", "install", "-y", "aria2"], capture_output=True).returncode != 0:
                subprocess.run(["apt-get", "-qq", "update"], capture_output=True)
                subprocess.run(["apt-get", "-qq", "install", "-y", "aria2"], capture_output=True)
        if shutil.which("aria2c"):
            r = subprocess.run(["aria2c", "-x", "4", "-s", "4", "-k", "8M", "--file-allocation=none", "--max-tries=30", "--retry-wait=10",
                                "--allow-overwrite=true", "--auto-file-renaming=false", "--console-log-level=warn", "--summary-interval=0",
                                "--download-result=hide", "-d", str(dest.parent), "-o", part.name, url])
            if r.returncode == 0 and part.exists():
                part.rename(dest)
                return
            print(f"  aria2c failed for {dest.name} (code {r.returncode}); one connection instead", flush=True)
            for f in (part, Path(str(part) + ".aria2")):
                f.unlink(missing_ok=True)
    download_one(url, dest)

In [ ]:
# 2 · Every number below was fixed in the rules before this notebook ran
RULES = "results/mimii_beats/RULES_domain_shift_2022_written_before_running.md"
# DCASE 2022 Task 2 on Zenodo: development, additional training and evaluation (MD5s read twice from the record pages)
ZD, ZX, ZE = "https://zenodo.org/records/6355122/files/", "https://zenodo.org/records/6462969/files/", "https://zenodo.org/records/6586456/files/"
TYPES = ["ToyCar", "ToyTrain", "bearing", "fan", "gearbox", "slider", "valve"]
DMD5 = {"ToyCar": "1ccca43751353e516ee7d6d55457c705", "ToyTrain": "17e5b8cae15ccb630a5c533551bc93b5", "bearing": "6381a00f9efc0ced779c8ad847e4ff59",
        "fan": "a1a9b488934a82426bacc933d87aacde", "gearbox": "c165dfef8c404256bd719c6fe1f7036f", "slider": "8c3a5466cf53e54872fd94998a67bfac",
        "valve": "1da37b2e82942dfba720984541e2ef60"}
XMD5 = {"ToyCar": "fde0d2e49a542c5e3e38f9ecd8329496", "ToyTrain": "e66a77da55eafe090f4f0645855fac2f", "bearing": "e5c41a4cddd1a5ccfe32fbd182e84d73",
        "fan": "ee94f9843b86550259a0a24e99fe02dc", "gearbox": "122bb02c6d32c281aae8bfcb3b6a2d1d", "slider": "9fe49f109f491471720762fa8eb366b6",
        "valve": "e2396a438d65e6c61fc0698df72cbf66"}
EMD5 = {"ToyCar": "3c52961e75d60623a40b9dcc47ec1009", "ToyTrain": "7df19e597a47b7cc1028969b6330be0f", "bearing": "47518a98084260c11fa61ce9a0c54b7c",
        "fan": "5b1398524c46a67badf5e94b903c3d84", "gearbox": "768c162d3d2bb5147127a2965d54c2ca", "slider": "b2901b49bc001d6446d470260d32db64",
        "valve": "372c723bb3112340a880c282f5171775"}
DC_DEV = {t_: (ZD + f"dev_{t_}.zip?download=1", f"dev_{t_}.zip", h) for t_, h in DMD5.items()}
XZIPS = {t_: (ZX + f"eval_data_{t_}_train.zip?download=1", f"eval_data_{t_}_train.zip", h) for t_, h in XMD5.items()}
DC_EVAL = {t_: (ZE + f"eval_data_{t_}_test.zip?download=1", f"eval_data_{t_}_test.zip", h) for t_, h in EMD5.items()}
# the evaluation set's ground truth and the official evaluator: one repository at one commit. EV_SHA is the script's SHA-256; GT_SHA is the
# SHA-256 of the 42 ground-truth files together (each file's path, a zero byte and its content, in path order)
EV_COMMIT = "de8719b7d35280ff3a39487c9d1003413eef7818"
EV_ARCHIVE = (f"https://github.com/Kota-Dohi/dcase2022_evaluator/archive/{EV_COMMIT}.zip", "dcase2022_evaluator.zip")
EV_SCRIPT, EV_SHA = "dcase2022_task2_evaluator.py", "31bae0d9fa981b4ed3e92414bf62cd06adf75e0afba3f1a2d3d4b9408c5fdf4a"
GT_SHA = "d61ad20294be2e127d53ec2bdc1e9f3d1fb4e5cb932f97137ba7e231f2c1f35e"
SECTIONS_DEV, SECTIONS_EVAL = ["section_00", "section_01", "section_02"], ["section_03", "section_04", "section_05"]
DEV_IDS, EVAL_IDS = {t_: list(SECTIONS_DEV) for t_ in TYPES}, {t_: list(SECTIONS_EVAL) for t_ in TYPES}   # a section is a machine
SR, LDN_FLOOR = 16000, 1e-6
NS_RULE = 10 * SR                                                     # every system reads the first 10 s of a clip
# Part 1 · the frozen recipe: notebook 23's lines, unchanged (notebook 21's settings; the track list fixes the random streams)
TRACKS = ["ARR", "MIC+", "MIC"]
ALL_TRACKS = list(TRACKS) + ["DC"]                                            # fixed order for random seeds
CFG = dict(lora_rank=16, lora_alpha=32, lora_dropout=0.1, emb_dim=256, subcenters=3, arc_scale=30.0, arc_margin=0.2,
           mixup_prob=0.5, mixup_beta=0.5, crop_s=4.0, lr_lora=2e-4, lr_head=1e-3, wd_head=1e-4,
           batch=32, epochs=10, warmup_epochs=1, clip_norm=5.0, xc_cap=600, pl_k=4, pl_starts=10, pl_seed=0,
           ts_prob=0.5, pd_balance_batches=8, se_seeds=[0, 1, 2], embed_batch=32)        # notebook 21's, unchanged
ECHO_HF, ECHO_SHA_N15 = "yucongzh/echo-small-0824", "1cc306d581c97604d70e451d24e8616b517f4dff5b4c488a7ea1bc860fbdafcc"
B3_SHA_N22B = "ba0f139a46b3a22f4ff78873db92214146cddfaad12db7c026f6dd758df1be6a"
ENCODERS = {"BEATs": {"name": "BEATs iter3+ (AS2M)", "dim": 768, "layers": 12},
            "BEATs-iter3": {"name": "BEATs iter3", "dim": 768, "layers": 12, "file": "BEATs_iter3.pt",
                            "sed_file": "BEATs_iter3_from_PretrainedSED.pt",          # Amendment 1: the third source
                            "sed": "https://github.com/fschmid56/PretrainedSED/releases/download/v0.0.1/BEATs_ssl.pt",
                            "share": "https://1drv.ms/u/s!AqeByhGUtINrgcpxJUNDxg4eU0r-vA?e=qezPJ5",
                            "azure": ("https://valle.blob.core.windows.net/share/BEATs/BEATs_iter3.pt?sv=2020-08-04"
                                      "&st=2023-03-01T07%3A51%3A05Z&se=2033-03-02T07%3A51%3A00Z&sr=c&sp=rl&sig=QJXmSJG9DbMKf48UDIU1MfzIro8HQOf3sqlNXiflY1I%3D")},
            "EAT-base": {"name": "EAT-base (epoch 30, pre-trained)", "dim": 768, "layers": 12, "repo": "worstchan/EAT-base_epoch30_pretrain"},
            "EAT-large": {"name": "EAT-large (epoch 20, pre-trained)", "dim": 1024, "layers": 24, "repo": "worstchan/EAT-large_epoch20_pretrain"}}
NEW_ENCODERS = ["BEATs-iter3"]                                       # notebook 23 needs only BEATs iter3 (MIC+ and MIC)
ENC_ORDER = list(ENCODERS)
EAT_NORM, EAT_FRAMES = (-4.268, 4.569), 1024                         # EAT's model card
GB_RATIO = 1.5
FROZEN = [("BEATs", "B", [0, 1, 2]), ("BEATs-iter3", "B3", [0, 1, 2])]   # MIC's final system: encoder, file code, seeds
# Part 2 · the second view: notebook 25's lines, unchanged
REF_COMMIT = "674c759bfaa7723619a4b1d8d00eb9d155c50ba6"
SV = dict(n_fft=2048, hop=512, n_mels=128, win_len=1024, c_dim=128, arc_s=30.0, arc_m=0.7, alpha=0.5, lr=1e-4, eta_min_ratio=0.1,
          epochs=300, batch=64, split_ratio=0.95, seeds=[0, 1, 2], sine_floor=1e-9, save_every=10, embed_batch=64,
          gpu_headroom_gb=5.0)
REF_FRAMES, REF_KERNEL, REF_CLASSES = 313, (8, 20), 42                # the reference's sizes for 10 s; 7 types × 6 sections
CAL_MIN = 30                                                          # a spread of form ZP needs at least 30 clips per type
MIN_SOURCE_FIRST = 600                                                # gate G0(b): a training section's first 600 clips by name are source-domain clips
GDEV_BAR = 0.55                                                       # gate G-DEV, on the development set's source-domain line: a system at chance has about 0.50
GDEV_SYSTEMS = ["ZP", "frozen", "view_paper"]                         # the kept system and its two parts
# the yardsticks: the challenge's team ranking, the official score (%) of the best system of each of its 31 teams (the results page, read twice)
TEAMS = {"Liu_CQUPT_task2_4": 70.97462888079512, "Kuroyanagi_NU-HDL_task2_2": 68.22312028564296, "Guan_HEU_task2_4": 68.0372804969291,
         "Deng_THU_task2_4": 67.62358685887781, "Venkatesh_MERL_task2_3": 67.56529995172603, "Wei_HEU_task2_2": 67.12235197698574,
         "Morita_SECOM_task2_1": 66.82507994516816, "Bai_JLESS_task2_1": 63.948583467398024, "Verbitskiy_DS_task2_1": 63.83127217235942,
         "Wilkinghoff_FKIE_task2_1": 63.724437030531156, "Nejjar_ETH_task2_1": 63.34598205922427, "Du_NERCSLIP_task2_1": 63.30309472782353,
         "Tozicka_NSW_task2_1": 62.50004793576084, "Yamashita_GU_task2_3": 62.387167480540676, "Almudevar_UZ_task2_2": 60.60499062666268,
         "Liu_BUPT_task2_2": 59.767720255838206, "Gou_UESTC_task2_4": 59.16904508333896, "PENG_NJUPT_task2_1": 58.47341574821306,
         "Li_CTRI_task2_1": 58.12785559862876, "Cohen_Technion_task2_2": 56.763333657296656, "Hu_NJU_task2_1": 55.07461026550378,
         "CHO_SG_task2_1": 54.468778744613665, "Narita_AIT_task2_2": 53.84966232209122, "Kodua_ITMO_task2_1": 52.96330674177412,
         "Jalalia_AIT_task2_1": 51.338525836051105, "Jinhyuk_SNU_task2_1": 51.02701922567823, "Siang_NTHU_task2_1": 50.59102827288451,
         "Kazakova_ITMO_task2_1": 49.89612147323122, "LEE_KNU_task2_1": 49.26732013587548, "Li_JAIST_task2": 48.93884539917759,
         "Zorin_AIRI_task2_1": 43.16915219364365}
BASELINES = {"DCASE2022_baseline_task2_MNV2": 54.01722421174122, "DCASE2022_baseline_task2_AE": 52.941609218413156}
# question 1's words, on scores in % of two decimals: above the first team; at least the 7th team's; at least the 16th team's (the middle of 31);
# at least the better baseline's
Q1_FIRST, Q1_LEADING, Q1_UPPER, Q1_BASE = 70.97, 66.83, 59.77, 54.02
Q3_BAR = 0.01                                                         # the fusion's words, on the official score
AE_EVAL_SHIFT = (64.23, 45.13)                                        # the autoencoder baseline on the evaluation set: harmonic mean AUC (source), AUC (target)
# the baselines' development tables (the task page; averages of five runs): per type, the harmonic means of AUC (source), AUC (target), pAUC, %
PUBLISHED_DEV = {"Autoencoder baseline": {"ToyCar": (90.41, 34.81, 52.74), "ToyTrain": (76.32, 23.35, 50.48), "bearing": (54.42, 58.38, 51.98),
                                          "fan": (78.59, 47.18, 57.52), "gearbox": (68.93, 62.64, 58.49), "slider": (77.95, 47.67, 55.78),
                                          "valve": (52.01, 49.46, 50.36)},
                 "MobileNetV2 baseline": {"ToyCar": (59.12, 51.96, 52.27), "ToyTrain": (57.26, 45.90, 51.52), "bearing": (60.58, 59.94, 57.14),
                                          "fan": (70.75, 48.22, 56.9), "gearbox": (69.21, 56.19, 56.03), "slider": (65.15, 38.23, 54.67),
                                          "valve": (67.09, 57.22, 62.42)}}
NAME = {"ToyCar": "ToyCar", "ToyTrain": "ToyTrain", "bearing": "bearing", "fan": "fan", "gearbox": "gearbox", "slider": "slide rail", "valve": "valve"}
EVAL_TOL, DRIVE_MIN_GB, DISK_MIN_GB = 1e-9, 3.0, 40.0
if TEST:
    _dc = json.loads(os.environ["P04_TEST_DCASE"])
    _c = json.loads(os.environ.get("P04_TEST_CFG", "{}"))
    TYPES = _c.get("types", TYPES)                                    # the synthetic sets may hold fewer machine types
    DC_DEV, XZIPS, DC_EVAL = ({t_: tuple(v) for t_, v in _dc[k_].items() if t_ in TYPES} for k_ in ("dev", "add", "eval"))
    EV_ARCHIVE, EV_SHA, GT_SHA = tuple(_dc["evaluator_archive"]), _dc["evaluator_sha256"], _dc["ground_truth_sha256"]
    DEV_IDS, EVAL_IDS = {t_: list(SECTIONS_DEV) for t_ in TYPES}, {t_: list(SECTIONS_EVAL) for t_ in TYPES}
    CFG.update(_c.get("cfg", {})); SV.update(_c.get("sv", {}))
    for k_, v_ in _c.get("encoders", {}).items():
        ENCODERS[k_].update(v_)
    CAL_MIN, NS_RULE, MIN_SOURCE_FIRST = _c.get("cal_min", CAL_MIN), _c.get("clip_samples", NS_RULE), _c.get("min_source_first", MIN_SOURCE_FIRST)
    GDEV_BAR, REF_CLASSES = _c.get("gdev_bar", GDEV_BAR), 6 * len(TYPES)
FROZEN_CAP = CFG["xc_cap"]                                            # the frozen recipe trains on the first 600 additional clips of a section
assert [sd for _, _, sd in FROZEN] == [CFG["se_seeds"]] * 2 and ENC_ORDER[:2] == [e for e, _, _ in FROZEN]
assert len(TEAMS) == 31 and sorted(TEAMS.values(), reverse=True) == list(TEAMS.values()), "the team ranking is not in order"
_ts = [round(v, 2) for v in TEAMS.values()]
assert (_ts[0], _ts[6], _ts[15], round(BASELINES["DCASE2022_baseline_task2_MNV2"], 2)) == (Q1_FIRST, Q1_LEADING, Q1_UPPER, Q1_BASE), "question 1's lines are not the table's"
print("rules:", RULES, "·", json.dumps(CFG), "·", json.dumps(SV))
if not TEST and DRIVE_FREE_GB < DRIVE_MIN_GB:
    raise SystemExit(f"stop: Drive has {DRIVE_FREE_GB:.1f} GB free and this run needs {DRIVE_MIN_GB} GB. Free some space in Google Drive, then run again.")
if not TEST and shutil.disk_usage(WORK).free / 1e9 < DISK_MIN_GB:
    raise SystemExit(f"stop: the local disk has {shutil.disk_usage(WORK).free / 1e9:.0f} GB free and this run needs {DISK_MIN_GB:.0f} GB")

In [ ]:
# 3 · DCASE 2022's own files of the seven types (MD5 against the Zenodo records), and every clip's first 10 s in one int16 store per
# type on the local disk; gates G0(a) and G0(b). The readers are notebook 23's (through notebook 25). The loader is notebook 27's with
# these files' names: a section where a machine ID was, the label and the domain read from the name.
CHECKED = {}                                        # files whose MD5 was already checked in this session


def get_file(entry):
    url, name, want = entry
    dest = WORK / name
    if CHECKED.get(name) == want and dest.exists():
        return dest
    if dest.exists() and md5(dest) == want:
        CHECKED[name] = want
        return dest
    t = time.time(); download(url, dest)
    got = md5(dest)
    print(f"{name}: downloaded in {time.time() - t:.0f} s · MD5 matches Zenodo: {got == want}", flush=True)
    if got != want:
        dest.unlink()
        raise SystemExit(f"stop: {name} is not the published file")
    CHECKED[name] = want
    return dest


def read_all(data):
    # WAV bytes (plain or WAVE_FORMAT_EXTENSIBLE, 16-bit PCM) -> (samples, channels) float32 in [-1, 1], sample rate
    assert data[:4] == b"RIFF" and data[8:12] == b"WAVE", "not a WAV file"
    pos, fmt, raw = 12, None, None
    while pos + 8 <= len(data):
        cid, size = data[pos:pos + 4], struct.unpack("<I", data[pos + 4:pos + 8])[0]
        body = data[pos + 8:pos + 8 + size]
        if cid == b"fmt ":
            tag, ch, sr, _, _, bits = struct.unpack("<HHIIHH", body[:16])
            if tag == 0xFFFE:
                tag = struct.unpack("<H", body[24:26])[0]
            fmt = (tag, ch, sr, bits)
        elif cid == b"data":
            raw = body
        pos += 8 + size + (size & 1)
    tag, ch, sr, bits = fmt
    assert tag == 1 and bits == 16, f"expected 16-bit PCM, got format {tag}, {bits}-bit"
    nf = len(raw) // (2 * ch)
    return np.frombuffer(raw[:nf * 2 * ch], dtype="<i2").reshape(-1, ch).astype(np.float32) / 32768.0, sr


def stream(fn, items, depth=8, workers=2):
    # fn(item) runs in background threads, results come back in order: unzipping overlaps the main thread's work
    with ThreadPoolExecutor(workers) as ex:
        q, it = deque(), iter(items)
        for item in it:
            q.append(ex.submit(fn, item))
            if len(q) >= depth:
                break
        while q:
            res = q.popleft().result()
            for item in it:
                q.append(ex.submit(fn, item))
                break
            yield res


def to_int16(v):
    # only for microphone-0 samples read from the 16-bit WAV files, so this conversion is exact
    return np.clip(np.round(v * 32768.0), -32768, 32767).astype(np.int16)


def f32(a):
    # a slice of a stream → float32 signal: int16 streams hold samples × 32768, float16 streams hold the signal itself
    a = np.asarray(a)
    return a.astype(np.float32) / 32768.0 if a.dtype == np.int16 else a.astype(np.float32)


STORE = WORK / "stores28"; STORE.mkdir(exist_ok=True)
DST, DMETA = {}, {}


def dcase_load(t):
    # every DCASE 2022 clip of type t in one int16 store, cut to its first NS_RULE samples (10 s, as the view's reference code cuts
    # it): development train and test, the additional training clips (all of them) and the evaluation test clips
    meta_f, arr_f = STORE / f"dc_{t}.json", STORE / f"dc_{t}.npy"
    if meta_f.exists() and arr_f.exists():
        DMETA[t] = json.loads(meta_f.read_text()); DST[t] = np.load(arr_f, mmap_mode="r")
        return
    zfs = {"dev": zipfile.ZipFile(get_file(DC_DEV[t])), "evaltrain": zipfile.ZipFile(get_file(XZIPS[t])), "evaltest": zipfile.ZipFile(get_file(DC_EVAL[t]))}
    rows, skipped = [], []
    for kind, zf in zfs.items():
        for nm in sorted(x for x in zf.namelist() if x.lower().endswith(".wav")):
            parts = nm.replace("\\", "/").split("/"); base, folder = parts[-1], parts[-2]
            if "__MACOSX" in parts or base.startswith("._"):          # an archive's metadata entry, not a clip (none is expected)
                skipped.append(nm); continue
            part = "train" if folder == "train" else "test"
            # the name of a training clip and of a development test clip carries its domain and its kind; an evaluation test clip's carries neither
            said = re.match(r"section_\d+_(source|target)_(train|test)_(normal|anomaly)_", base)
            label = (1 if (said and said.group(3) == "anomaly") else 0) if (kind == "dev" and part == "test") else (0 if part == "train" else -1)
            dom = said.group(1) if said else ""
            rows.append((kind, nm, "dev" if kind == "dev" else "eval", part, re.search(r"(section_\d+)", base).group(1), label, base, dom))
    ids = {(s_, p_): sorted(set(r[4] for r in rows if r[2] == s_ and r[3] == p_)) for s_ in ("dev", "eval") for p_ in ("train", "test")}
    if not (ids[("dev", "train")] == ids[("dev", "test")] == DEV_IDS[t] and ids[("eval", "train")] == ids[("eval", "test")] == EVAL_IDS[t]):
        raise SystemExit(f"stop: G0(b) failed. The sections of {t} are not the rules': {ids}")
    assert not any(r[2] == "eval" and r[3] == "train" and "_train_normal_" not in r[6] for r in rows), "the additional training clips are normal"
    ns, lengths = NS_RULE, Counter()
    tmp = arr_f.with_name(arr_f.stem + ".part.npy")
    mm = np.lib.format.open_memmap(tmp, mode="w+", dtype=np.int16, shape=(len(rows), ns))
    for i, (x, sr) in enumerate(stream(lambda i: read_all(zfs[rows[i][0]].read(rows[i][1])), range(len(rows)))):
        if not (sr == SR and x.shape[1] == 1 and x.shape[0] >= ns):
            raise SystemExit(f"stop: G0(b) failed. {rows[i][1]} is not a 16 kHz, one-channel clip of at least {ns / SR:g} s "
                             f"(rate {sr}, channels {x.shape[1]}, samples {x.shape[0]})")
        lengths[(f"{rows[i][2]}|{rows[i][3]}", int(x.shape[0]))] += 1
        mm[i] = to_int16(x[:ns, 0])
    mm.flush(); del mm
    os.replace(tmp, arr_f)
    for zf in zfs.values():
        zf.close()
    DMETA[t] = {"path": [r[1] for r in rows], "name": [r[6] for r in rows], "set": [r[2] for r in rows], "part": [r[3] for r in rows],
                "mid": [r[4] for r in rows], "label": [r[5] for r in rows], "domain": [r[7] for r in rows], "samples": int(ns), "skipped_archive_entries": skipped,
                "lengths": {k: {str(n_): int(c) for (k2, n_), c in sorted(lengths.items()) if k2 == k} for k in sorted(set(k_ for k_, _ in lengths))},
                "zips": {"dev": list(DC_DEV[t][1:]), "additional": list(XZIPS[t][1:]), "evaluation": list(DC_EVAL[t][1:])}}
    meta_f.write_text(json.dumps(DMETA[t]))
    DST[t] = np.load(arr_f, mmap_mode="r")
    if IN_COLAB:
        for z in (DC_DEV[t], XZIPS[t], DC_EVAL[t]):
            (WORK / z[1]).unlink(missing_ok=True)


def dc_rows(t, set_, part, q=None):
    m = DMETA[t]
    return np.array([i for i in range(len(m["name"])) if m["set"][i] == set_ and m["part"][i] == part and (q is None or m["mid"][i] == q)], dtype=np.int64)


t_data = time.time()
DCASE = {"counts": {}, "lengths": {}, "domains": {}}
for t in TYPES:
    dcase_load(t)
    _m = DMETA[t]; _mid, _dom = np.array(_m["mid"]), np.array(_m["domain"])
    DCASE["counts"][t] = {f"{s_}|{p_}": {q: int(len(dc_rows(t, s_, p_, q))) for q in sorted(set(_mid[dc_rows(t, s_, p_)].tolist()))}
                          for s_ in ("dev", "eval") for p_ in ("train", "test")}
    DCASE["domains"][t] = {f"{s_}|{p_}": {q: {d_: int((_dom[dc_rows(t, s_, p_, q)] == d_).sum()) for d_ in ("source", "target")}
                                          for q in sorted(set(_mid[dc_rows(t, s_, p_)].tolist()))} for s_, p_ in (("dev", "train"), ("dev", "test"), ("eval", "train"))}
    DCASE["lengths"][t] = _m["lengths"]
    print(f"DCASE 2022 · {t}: {len(_m['name'])} clips · training clips per section (source, target): "
          f"{({q: (v['source'], v['target']) for k_ in ('dev|train', 'eval|train') for q, v in DCASE['domains'][t][k_].items()})} · test clips per section: "
          f"{({**DCASE['counts'][t]['dev|test'], **DCASE['counts'][t]['eval|test']})} · samples per clip before the cut (set|part: samples: clips): {DCASE['lengths'][t]}", flush=True)
NS = int(DMETA[TYPES[0]]["samples"])
assert all(int(DMETA[t]["samples"]) == NS for t in TYPES) and NS == NS_RULE, "stop: a store on the local disk was not cut to the rules' length"

# G0(a) · every zip is the published file: get_file stops on another MD5, and a store keeps the names and MD5s of the zips it was built from
_want = {t: {"dev": list(DC_DEV[t][1:]), "additional": list(XZIPS[t][1:]), "evaluation": list(DC_EVAL[t][1:])} for t in TYPES}
G0 = {"a": {"ok": bool(all(DMETA[t]["zips"] == _want[t] for t in TYPES)), "zips": _want}}
print(f"G0(a) · the {3 * len(TYPES)} zips are the published files (MD5):", G0["a"]["ok"])
if not G0["a"]["ok"]:
    raise SystemExit(f"stop: G0(a) failed. A store in {STORE} was built from other files; delete that folder and run again")


def g0b(t):
    # the files are what the rules assume (16 kHz, one channel and 10 s were checked per file by the loader)
    m = DMETA[t]; nm, st, pt, lb, mid, dom = (np.array(m[k]) for k in ("name", "set", "part", "label", "mid", "domain"))
    c = DCASE["counts"][t]; tr, ev_test = pt == "train", (st == "eval") & (pt == "test")
    first_target = {}                                                 # per training section: how many source-domain clips come before its first target-domain clip by name
    for s_, ids in (("dev", DEV_IDS[t]), ("eval", EVAL_IDS[t])):
        for q in ids:
            rows = dc_rows(t, s_, "train", q); d_ = dom[rows[np.argsort(nm[rows], kind="stable")]]
            first_target[q] = int(np.argmax(d_ == "target")) if (d_ == "target").any() else -1
    both = all(int(((st == "dev") & (pt == "test") & (mid == q) & (lb == k_) & (dom == d_)).sum()) > 0 for q in DEV_IDS[t] for k_ in (0, 1) for d_ in ("source", "target"))
    numbered = all(sorted(int(re.search(r"(\d{4})\.wav$", n_).group(1)) for n_ in nm[ev_test & (mid == q)]) == list(range(int((ev_test & (mid == q)).sum()))) for q in EVAL_IDS[t])
    out = {"sections": bool(sorted(c["dev|train"]) == sorted(c["dev|test"]) == DEV_IDS[t] and sorted(c["eval|train"]) == sorted(c["eval|test"]) == EVAL_IDS[t]),
           "training_clips_are_normal": bool(all("_train_normal_" in n_ for n_ in nm[tr]) and (lb[tr] == 0).all()),
           "training_clips_carry_their_domain": bool(np.isin(dom[tr], ["source", "target"]).all()),
           "source_clips_come_first_by_name": bool(all(v >= MIN_SOURCE_FIRST for v in first_target.values())),
           "every_training_section_has_target_clips": bool(all(v >= 0 for v in first_target.values())),
           "development_sections_have_both_kinds_in_both_domains": bool(both),
           "development_test_names_carry_label_and_domain": bool(all(("_test_normal_" in n_ or "_test_anomaly_" in n_) and ("_source_" in n_ or "_target_" in n_)
                                                                     for n_ in nm[(st == "dev") & (pt == "test")])),
           "evaluation_test_names_carry_no_label_or_domain": bool(ev_test.any() and not any(w in n_ for n_ in nm[ev_test] for w in ("normal", "anomaly", "source", "target"))
                                                                  and (dom[ev_test] == "").all()),
           "evaluation_test_clips_are_numbered_from_0000": bool(numbered),
           "no_evaluation_label_in_the_store": bool((lb[ev_test] == -1).all()),
           "names_are_unique_in_each_folder": bool(all(len(set(nm[(st == s_) & (pt == p_)].tolist())) == int(((st == s_) & (pt == p_)).sum())
                                                       for s_ in ("dev", "eval") for p_ in ("train", "test"))),
           "clips": int(len(nm)), "skipped_archive_entries": int(len(m["skipped_archive_entries"])), "source_clips_before_the_first_target_clip": first_target,
           "shortest_clip_samples": int(min(int(k_) for d_ in m["lengths"].values() for k_ in d_)),
           "longest_clip_samples": int(max(int(k_) for d_ in m["lengths"].values() for k_ in d_))}
    out["ok"] = bool(all(v for k, v in out.items() if isinstance(v, bool)) and out["shortest_clip_samples"] >= NS)
    return out


G0["b"] = {t: g0b(t) for t in TYPES}
print(f"G0(b) · the sections, the clip kinds, the domains and the names are the rules'; every clip is 16 kHz, one channel, at least {NS / SR:g} s:",
      all(v["ok"] for v in G0["b"].values()), {t: {k: v for k, v in d.items() if v is False} or "all checks true" for t, d in G0["b"].items()})
if not all(v["ok"] for v in G0["b"].values()):
    raise SystemExit(f"stop: G0(b) failed, the files are not what the rules assume: {G0['b']}")
N_TRAIN_ALL, N_TEST_ALL = (int(sum(sum(DCASE["counts"][t][f"{s_}|{p_}"].values()) for t in TYPES for s_ in ("dev", "eval"))) for p_ in ("train", "test"))
PREP_S = round(time.time() - t_data, 1)
print(f"data ready · {N_TRAIN_ALL} training clips and {N_TEST_ALL} test clips · {NS} samples per clip are kept ({NS / SR:g} s) · {PREP_S / 60:.1f} min")

In [ ]:
# 4 · Notebook 28's saved files (gate G-FILES), and the system as it is, by notebook 28's own functions. Nothing is trained
OUT29 = OUT / "n29_backends"; OUT29.mkdir(exist_ok=True)
_need = ([f"emb_{c}_DC_XC_s{s}_f0.npz" for _, c, sd in FROZEN for s in sd] + [f"dcase/echo_{t}.npy" for t in TYPES] + [f"emb_SV_s{s}.npz" for s in SV["seeds"]]
         + ["hold_out.json", "domain_shift.json", "dcase2022_evaluation/scores_written.json"] + [f"development_scores/scores_{t}_{q}.csv" for t in TYPES for q in DEV_IDS[t]])
_missing = [f for f in _need if not (OUT / f).exists()]
if _missing:
    raise SystemExit(f"stop: G-FILES failed. Notebook 28's files are not all in {OUT} (for example {_missing[0]}). Run notebook 28 first.")
ECHO_DC = {t: np.load(OUT / "dcase" / f"echo_{t}.npy") for t in TYPES}
if any(len(ECHO_DC[t]) != len(DMETA[t]["name"]) for t in TYPES):
    raise SystemExit("stop: G-FILES failed. A saved file does not have this session's number of clips: this folder is not from these recordings")
SEEDS = list(SV["seeds"])
def unit(E):
    return E / np.linalg.norm(E, axis=-1, keepdims=True)


def b1(R, X):
    return 1.0 - (unit(X) @ unit(R).T).max(1)


def ldn1(R, X):
    # Wilkinghoff et al. (TASLP 2026), ratio form with K = 1: min over references y of d(x, y) / d(y, y₁)
    U = unit(R); Srr = U @ U.T; np.fill_diagonal(Srr, -np.inf)
    d1 = np.maximum(1.0 - Srr.max(1), LDN_FLOOR)
    return ((1.0 - unit(X) @ U.T) / d1[None, :]).min(1)


def dc_metrics(y, s):
    # as the official evaluator: sklearn's AUC, and its pAUC with max_fpr = 0.1 (standardised)
    return {"AUC": float(roc_auc_score(y, s)), "pAUC": float(roc_auc_score(y, s, max_fpr=0.1)), "n_test": int(len(y)), "n_anomalies": int(np.sum(y))}


FM = {}


def frozen_matrix(code, seeds, t):
    k = (code, t)
    if k not in FM:
        Es = [np.load(OUT / f"emb_{code}_DC_XC_s{s}_f0.npz", allow_pickle=False)[f"F__{t}"].astype(np.float32) for s in seeds]
        E = np.concatenate([unit(e) for e in Es], 1) / math.sqrt(len(Es))
        FM[k] = np.concatenate([E, unit(ECHO_DC[t].astype(np.float64))], 1) / math.sqrt(2.0)
        assert len(FM[k]) == len(DMETA[t]["name"]), f"Part 1's embeddings of {t} do not have this session's number of clips"
    return FM[k]


def machines(t, set_):
    return sorted(set(np.array(DMETA[t]["mid"])[dc_rows(t, set_, "test")].tolist()))


def frozen_score(t, ref, x):
    return sum(ldn1(frozen_matrix(c, sd, t)[ref], frozen_matrix(c, sd, t)[x]) for _, c, sd in FROZEN) / len(FROZEN)


def spread(logs):
    # the centre and the robust spread (1.4826 × the median absolute deviation) of a system's log scores on normal clips
    if len(logs) == 0:
        return {"n": 0, "median": None, "sigma": None}
    med = float(np.median(logs))
    return {"n": int(len(logs)), "median": med, "sigma": float(1.4826 * np.median(np.abs(logs - med)))}


def part_score(code, t, ref, x):
    # one encoder's part of MIC's scorer on its own: its seeds' F with ECHO concatenated (the frozen system is the mean of the two parts)
    sd = next(s for _, c, s in FROZEN if c == code)
    return ldn1(frozen_matrix(code, sd, t)[ref], frozen_matrix(code, sd, t)[x])


def echo_score(t, ref, x):
    # ECHO alone: global LDN1 on its frozen embedding
    E = ECHO_DC[t].astype(np.float64)
    return ldn1(E[ref], E[x])


SF = {}                                                               # (type, set, machine) → the frozen system's scores of its test clips


def frozen_of(t, set_, q):
    # computed once per machine; the references are all of the machine's normal training clips
    k = (t, set_, q)
    if k not in SF:
        SF[k] = frozen_score(t, dc_rows(t, set_, "train", q), dc_rows(t, set_, "test", q))
    return SF[k]


_R, _X = np.random.default_rng(11).standard_normal((40, 9)), np.random.default_rng(12).standard_normal((7, 9))
assert np.allclose(ldn1(_R, _X), [min((1 - unit(x) @ unit(r)) / max(1 - max(unit(r) @ unit(r2) for j, r2 in enumerate(_R) if j != i), LDN_FLOOR)
                                      for i, r in enumerate(_R)) for x in _X]), "stop: LDN1 is not the rules' ratio"

t_f = time.time()
LAB = {t: np.array(DMETA[t]["label"]) for t in TYPES}
CAL_F, CALF_SC = {}, {}
for t in TYPES:
    # σ_F (notebook 25's rule): the frozen system's log score on the additional-training clips that its trainings did not use (those
    # after the first FROZEN_CAP of an evaluation section, by name), against the clips they did use. No test clip.
    _nm, _logs, CALF_SC[t] = np.array(DMETA[t]["name"]), [], []
    for q in machines(t, "eval"):
        rows = dc_rows(t, "eval", "train", q); rows = rows[np.argsort(_nm[rows], kind="stable")]
        if len(rows) > FROZEN_CAP:
            _s = frozen_score(t, rows[:FROZEN_CAP], rows[FROZEN_CAP:])
            _logs.append(np.log(np.maximum(_s, LDN_FLOOR))); CALF_SC[t].extend(zip(_nm[rows[FROZEN_CAP:]].tolist(), _s.tolist()))
    CAL_F[t] = spread(np.concatenate(_logs) if _logs else np.zeros(0))
    FM.clear(); gc.collect()
print("form ZP · the frozen system's spread on normal clips it did not train on (clips, σ of the log score):",
      {t: (CAL_F[t]["n"], None if CAL_F[t]["sigma"] is None else round(CAL_F[t]["sigma"], 4)) for t in TYPES}, f"· {time.time() - t_f:.0f} s")


def training_lists():
    # one class per machine: for each type, its development machines (all train/ clips of the development zip), then its evaluation
    # machines (all clips of the additional training zip); items (type index, row in the type's store, class)
    items, classes, cls_of = [], [], {}
    for ti, t in enumerate(TYPES):
        nm, mids = np.array(DMETA[t]["name"]), np.array(DMETA[t]["mid"])
        for set_, what in (("dev", "development"), ("eval", "additional")):
            for q in sorted(set(mids[dc_rows(t, set_, "train")].tolist())):
                rows = dc_rows(t, set_, "train", q); rows = rows[np.argsort(nm[rows], kind="stable")]
                c = len(classes); classes.append(f"{t}|{q}|DCASE2022-{what}"); cls_of[(t, q)] = c
                items.extend((ti, int(r), c) for r in rows)
    return np.array(items, dtype=np.int64), classes, cls_of


VIEWS, CES, VM = {}, {}, {}                                           # seed → type → features (n, 128) and the paper's score (n,)
HOLD_ROWS, CAL_P = {}, {}                                             # type → store rows of the held-out clips; (seeds, type) → the view's spread
SYSTEMS = ["ZP", "frozen", "view_paper", "view_LDN1", "view_B1", "frozen_B", "frozen_B3", "echo"]
LABEL = {"ZP": "kept system · frozen + view (ZP)", "frozen": "frozen recipe alone", "view_paper": "view alone · paper's score",
         "view_LDN1": "view alone · LDN1", "view_B1": "view alone · B1", "frozen_B": "frozen · BEATs iter3+ part",
         "frozen_B3": "frozen · BEATs iter3 part", "echo": "ECHO alone · LDN1"}
VALUES = ("AUC_source", "AUC_target", "pAUC")                         # a section's three values, as the official evaluator names them


def view_matrix(seeds, t):
    # the program's seed ensemble: each seed's features unit-normalised, concatenated, ÷√seeds
    k = (tuple(seeds), t)
    if k not in VM:
        VM[k] = np.concatenate([unit(VIEWS[s][t]) for s in seeds], 1) / math.sqrt(len(seeds))
    return VM[k]


def calibrate_paper(seeds, t):
    # form ZP's calibration of the view: the log of the paper's score (averaged over the seeds) on the held-out clips. No test clip.
    k = (tuple(seeds), t)
    if k not in CAL_P:
        rows = np.asarray(HOLD_ROWS[t])
        CAL_P[k] = spread(np.log(np.maximum(sum(CES[s][t][rows] for s in seeds) / len(seeds), LDN_FLOOR)) if len(rows) else np.zeros(0))
    return CAL_P[k]


def paper(seeds, t, rows):
    # the paper's score (the cross-entropy for the clip's own section class, margin on), averaged over the seeds
    return sum(CES[s][t][rows] for s in seeds) / len(seeds)


def zp_ok(seeds):
    # form ZP needs a spread from at least CAL_MIN clips, for both systems and every type
    return all(c["n"] >= CAL_MIN and c["sigma"] is not None and c["sigma"] > 1e-9 for t in TYPES for c in (CAL_F[t], calibrate_paper(seeds, t)))


def score_of(name, t, set_, q, seeds):
    # one system's anomaly scores of one section's test clips; the references are all of the section's normal training clips
    ref, te = dc_rows(t, set_, "train", q), dc_rows(t, set_, "test", q)
    if name == "frozen":
        return frozen_of(t, set_, q)
    if name == "view_paper":
        return paper(seeds, t, te)
    if name == "ZP":                                                  # each system's log score in units of its spread on unseen normal clips
        return (np.log(np.maximum(frozen_of(t, set_, q), LDN_FLOOR)) / CAL_F[t]["sigma"]
                + np.log(np.maximum(paper(seeds, t, te), LDN_FLOOR)) / calibrate_paper(seeds, t)["sigma"])
    if name in ("frozen_B", "frozen_B3"):
        return part_score(name.split("_")[1], t, ref, te)
    if name == "echo":
        return echo_score(t, ref, te)
    V = view_matrix(seeds, t)
    if name == "view_LDN1":
        return ldn1(V[ref], V[te])
    if name == "view_B1":
        return b1(V[ref], V[te])
    raise ValueError(name)


def hm(v):
    # the harmonic mean, as the official evaluator takes it
    v = np.maximum(np.asarray(v, dtype=np.float64), sys.float_info.epsilon)
    return float(len(v) / np.sum(1.0 / v))


def section_metrics(y, s, dom):
    # the official evaluator's values of one section. AUC of a domain: that domain's normal clips against all the section's anomalous
    # clips, whatever their domain. pAUC: all the section's clips, max_fpr = 0.1
    y, s, dom = np.asarray(y), np.asarray(s, dtype=np.float64), np.asarray(dom)
    out = {"n_test": int(len(y)), "n_anomalies": int(y.sum())}
    for d in ("source", "target"):
        k = (dom == d) | (y != 0)
        out[f"AUC_{d}"] = float(roc_auc_score(y[k], s[k])); out[f"n_normal_{d}"] = int(((dom == d) & (y == 0)).sum())
    out["pAUC"] = float(roc_auc_score(y, s, max_fpr=0.1)); out["AUC_all"] = float(roc_auc_score(y, s))
    return out


def summary(per):
    # per: type → section → its values. The type's score is the harmonic mean of its sections' three values; the official score Ω is the
    # harmonic mean of every section's three values; the source, target and pAUC lines are harmonic means over the sections
    types = {t: {"score": hm([d[q][v] for q in d for v in VALUES]), **{v: hm([d[q][v] for q in d]) for v in VALUES}, "per_section": d} for t, d in per.items()}
    return {"omega": hm([d[q][v] for d in per.values() for q in d for v in VALUES]), **{v: hm([d[q][v] for d in per.values() for q in d]) for v in VALUES},
            "values": int(sum(len(d) * len(VALUES) for d in per.values())), "per_type": types}


def dev_eval(seeds, names, keep=None):
    # the development set under each system. Its labels and domains are in the file names. With `keep`, every clip's scores are kept too
    names = [nm for nm in names if nm != "ZP" or zp_ok(seeds)]            # the calibrated form only when its spreads can be formed
    out = {}
    for nm in names:
        per = {}
        for t in TYPES:
            m = DMETA[t]; lab, dom, fn = np.array(m["label"]), np.array(m["domain"]), np.array(m["name"])
            for q in machines(t, "dev"):
                te = dc_rows(t, "dev", "test", q); sc = score_of(nm, t, "dev", q, seeds)
                per.setdefault(t, {})[q] = section_metrics(lab[te], sc, dom[te])
                if keep is not None:
                    keep.setdefault((t, q), {"file": fn[te].tolist(), "label": lab[te].tolist(), "domain": dom[te].tolist()})[nm] = np.asarray(sc, dtype=np.float64)
        out[nm] = summary(per)
    FM.clear(); VM.clear(); gc.collect()
    return out


def fusion_word(kept, frozen, view):
    # the rules' three words for the fusion, on official scores of four decimals: against the better of its two parts
    d = round(round(kept, 4) - max(round(frozen, 4), round(view, 4)), 4)
    return ("helps" if d >= Q3_BAR else "costs" if d < -Q3_BAR else "no clear difference"), d


def fusion_table(res):
    # the kept system's official score less the frozen recipe's and less the view's, and the word
    k, f, v = (res[nm]["omega"] for nm in ("ZP", "frozen", "view_paper")); w, d = fusion_word(k, f, v)
    return {"kept": round(k, 4), "frozen": round(f, 4), "view": round(v, 4), "kept_less_frozen": round(round(k, 4) - round(f, 4), 4),
            "kept_less_view": round(round(k, 4) - round(v, 4), 4), "kept_less_better_part": d, "word": w}


def shift_table(res):
    # question 2: per system, the harmonic means over the sections of AUC (source) and AUC (target), and the difference
    return {nm: {"AUC_source": round(res[nm]["AUC_source"], 4), "AUC_target": round(res[nm]["AUC_target"], 4),
                 "source_less_target": round(round(res[nm]["AUC_source"], 4) - round(res[nm]["AUC_target"], 4), 4)} for nm in res}


assert [fusion_word(0.65, 0.64, 0.60)[0], fusion_word(0.6499, 0.64, 0.60)[0], fusion_word(0.63, 0.64, 0.60)[0], fusion_word(0.6299, 0.64, 0.60)[0]] == \
    ["helps", "no clear difference", "no clear difference", "costs"], "stop: the fusion's words are not the rules'"
# the metrics on a hand-made section: four normal clips (two per domain) and two anomalous ones
_m = section_metrics([0, 0, 0, 0, 1, 1], [0.1, 0.4, 0.3, 0.9, 0.5, 0.8], ["source", "source", "target", "target", "source", "target"])
assert (_m["AUC_source"], _m["AUC_target"], _m["AUC_all"]) == (1.0, 0.5, 0.75) and abs(hm([0.5, 1.0]) - 2 / 3) < 1e-12, "stop: the metrics are not the evaluator's"


def show(res, names, title):
    print(title)
    print(f"  {'':<36}{'Ω':>9}{'source':>9}{'target':>9}{'pAUC':>9}   " + "".join(f"{NAME[t][:10]:>11}" for t in TYPES))
    for nm in [n_ for n_ in names if n_ in res]:
        r = res[nm]
        print(f"  {LABEL[nm]:<36}{r['omega']:>9.4f}{r['AUC_source']:>9.4f}{r['AUC_target']:>9.4f}{r['pAUC']:>9.4f}   " + "".join(f"{r['per_type'][t]['score']:>11.4f}" for t in TYPES))
ITEMS, CLASSES, CLS_OF = training_lists()
_h = json.loads((OUT / "hold_out.json").read_text())
if _h["n_clips"] != len(ITEMS):
    raise SystemExit("stop: G-FILES failed. The saved hold-out belongs to other training lists")
HOLD = np.array(_h["hold_out"], dtype=np.int64)
HOLD_ROWS.update({t: ITEMS[HOLD][ITEMS[HOLD][:, 0] == ti][:, 1] for ti, t in enumerate(TYPES)})
for _s in SEEDS:
    _D = np.load(OUT / f"emb_SV_s{_s}.npz", allow_pickle=False)
    VIEWS[_s] = {t: _D[f"V__{t}"].astype(np.float64) for t in TYPES}; CES[_s] = {t: _D[f"CE__{t}"].astype(np.float64) for t in TYPES}
GFILES = {"files": len(_need), "clips": {t: len(DMETA[t]["name"]) for t in TYPES},
          "ok": bool(all(len(VIEWS[s][t]) == len(DMETA[t]["name"]) == len(ECHO_DC[t]) for s in SEEDS for t in TYPES))}
print("G-FILES · notebook 28's saved files are there and have this session's number of clips:", GFILES["ok"], GFILES["clips"])
if not GFILES["ok"]:
    raise SystemExit("stop: G-FILES failed. A saved file does not have this session's number of clips")
if not zp_ok(SEEDS):
    raise SystemExit("stop: a spread of form ZP cannot be formed")
DOM = {t: np.array(DMETA[t]["domain"]) for t in TYPES}

In [ ]:
# 5 · Gate G-SAME: the system as it is, recomputed here from the saved embeddings, gives notebook 28's saved scores on every development
# test clip and every evaluation test clip. That proves this session's rows are the clips notebook 28 scored. No label is read here.
SAME_TOL = 1e-6
ASIS = {"ZP": "kept", "frozen": "frozen", "view_paper": "view"}       # system → notebook 28's folder of evaluation scores
_worst, _n = {nm: 0.0 for nm in ASIS}, 0
_rel = lambda a, b: float(np.max(np.abs(a - b) / np.maximum(np.abs(b), 1e-12))) if len(a) else 0.0
for t in TYPES:
    _nm = np.array(DMETA[t]["name"])
    for q in machines(t, "dev"):
        te = dc_rows(t, "dev", "test", q)
        with open(OUT / "development_scores" / f"scores_{t}_{q}.csv", newline="") as fh:
            saved = {r["file"]: r for r in csv.DictReader(fh)}
        if sorted(saved) != sorted(_nm[te].tolist()):
            raise SystemExit(f"stop: G-SAME failed. The development clips of {t} {q} are not the ones notebook 28 scored")
        for nm in ASIS:
            _worst[nm] = max(_worst[nm], _rel(score_of(nm, t, "dev", q, SEEDS), np.array([float(saved[f][nm]) for f in _nm[te]])))
        _n += len(te)
    for q in machines(t, "eval"):
        te = dc_rows(t, "eval", "test", q)
        for nm, folder in ASIS.items():
            saved = dict(ln.split(",") for ln in (OUT / "dcase2022_evaluation" / folder / f"anomaly_score_{t}_{q}_test.csv").read_text().splitlines() if ln)
            if sorted(saved) != sorted(_nm[te].tolist()):
                raise SystemExit(f"stop: G-SAME failed. The evaluation clips of {t} {q} are not the ones notebook 28 scored")
            _worst[nm] = max(_worst[nm], _rel(score_of(nm, t, "eval", q, SEEDS), np.array([float(saved[f]) for f in _nm[te]])))
        _n += len(te)
    FM.clear(); VM.clear(); gc.collect()
GSAME = {"clips": int(_n), "largest_relative_difference": _worst, "tolerance": SAME_TOL, "ok": bool(max(_worst.values()) <= SAME_TOL)}
print(f"G-SAME · the system as it is gives notebook 28's saved scores on {_n} test clips (largest relative difference "
      f"{max(_worst.values()):.1e}, allowed {SAME_TOL:.0e}):", GSAME["ok"])
if not GSAME["ok"]:
    raise SystemExit("stop: G-SAME failed. The saved embeddings do not reproduce notebook 28's scores")

In [ ]:
# 6 · The three scorings, the six candidates, the development set, and the choice by the rules
from sklearn.cluster import KMeans

KM_CENTRES, ADOPT_BAR = 16, 0.01
BACKENDS = ["DN", "KM", "NN"]
CANDS = [f"{k}_{b}" for k in ("K", "F") for b in BACKENDS]            # the rules' order: K_DN, K_KM, K_NN, F_DN, F_KM, F_NN


def nearest(R, X):
    # the cosine distance from each row of X to its nearest row of R
    return 1.0 - (unit(X) @ unit(R).T).max(1)


def own_level(R):
    # the median, over the clips of R, of the distance to their nearest other clip of R
    U = unit(R); S = U @ U.T; np.fill_diagonal(S, -np.inf)
    return max(float(np.median(1.0 - S.max(1))), LDN_FLOOR)


def backend(b, M, rs, rt, x):
    # one scoring of the clips x of matrix M; rs and rt are the rows of the source-domain and the target-domain training clips
    if b == "NN":
        return nearest(M[np.concatenate([rs, rt])], M[x])
    if b == "DN":
        out = nearest(M[rs], M[x]) / own_level(M[rs])
        return np.minimum(out, nearest(M[rt], M[x]) / own_level(M[rt])) if len(rt) >= 2 else out
    if b == "KM":
        U = unit(M[rs])
        C = unit(KMeans(n_clusters=KM_CENTRES, n_init=10, random_state=0).fit(U).cluster_centers_) if len(rs) > KM_CENTRES else U
        out = nearest(C, M[x])
        return np.minimum(out, nearest(M[rt], M[x])) if len(rt) else out
    raise ValueError(b)


def by_domain(t, ref):
    ref = np.asarray(ref)
    return ref[DOM[t][ref] == "source"], ref[DOM[t][ref] == "target"]


def frozen_b(b, t, ref, x):
    # the frozen recipe under scoring b: the mean of its two encoder parts
    rs, rt = by_domain(t, ref)
    return sum(backend(b, frozen_matrix(c, sd, t), rs, rt, x) for _, c, sd in FROZEN) / len(FROZEN)


def view_b(b, t, ref, x):
    rs, rt = by_domain(t, ref)
    return backend(b, view_matrix(SEEDS, t), rs, rt, x)


t_c = time.time()
_assert_R = np.random.default_rng(29).standard_normal((60, 7))
assert np.allclose(backend("NN", _assert_R, np.arange(40), np.arange(40, 50), np.arange(50, 60)),
                   [min(1 - unit(x) @ unit(r) for r in _assert_R[:50]) for x in _assert_R[50:]]), "stop: NN is not the rules' distance"
assert np.allclose(backend("DN", _assert_R, np.arange(40), np.arange(40, 50), np.arange(50, 60)),
                   [min(min(1 - unit(x) @ unit(r) for r in _assert_R[a:b_]) / np.median([min(1 - unit(r) @ unit(r2) for j, r2 in enumerate(_assert_R[a:b_]) if j != i)
                                                                                     for i, r in enumerate(_assert_R[a:b_])]) for a, b_ in ((0, 40), (40, 50)))
                    for x in _assert_R[50:]]), "stop: DN is not the rules' ratio"
# the two spreads of form ZP under each scoring. No test clip.
CAL29, CAL29_SC = {b: {"frozen": {}, "view": {}} for b in BACKENDS}, {}
for t in TYPES:
    _nm, _mid, _set, _part = np.array(DMETA[t]["name"]), np.array(DMETA[t]["mid"]), np.array(DMETA[t]["set"]), np.array(DMETA[t]["part"])
    hold = np.asarray(HOLD_ROWS[t], dtype=np.int64)
    for b in BACKENDS:
        logs, kept = [], []
        for q in machines(t, "eval"):                                  # σ_F: the clips after the first FROZEN_CAP of a section by name, against those
            rows = dc_rows(t, "eval", "train", q); rows = rows[np.argsort(_nm[rows], kind="stable")]
            if len(rows) > FROZEN_CAP:
                s = frozen_b(b, t, rows[:FROZEN_CAP], rows[FROZEN_CAP:])
                logs.append(np.log(np.maximum(s, LDN_FLOOR))); kept += list(zip(_nm[rows[FROZEN_CAP:]].tolist(), s.tolist()))
        CAL29[b]["frozen"][t] = spread(np.concatenate(logs) if logs else np.zeros(0)); CAL29_SC[(b, "frozen", t)] = kept
        logs, kept = [], []
        for set_ in ("dev", "eval"):                                   # σ_P: the held-out clips, each against its section's other training clips
            for q in machines(t, set_):
                tr = dc_rows(t, set_, "train", q); h = np.intersect1d(tr, hold); ref = np.setdiff1d(tr, hold)
                if len(h):
                    s = view_b(b, t, ref, h)
                    logs.append(np.log(np.maximum(s, LDN_FLOOR))); kept += list(zip(_nm[h].tolist(), s.tolist()))
        CAL29[b]["view"][t] = spread(np.concatenate(logs) if logs else np.zeros(0)); CAL29_SC[(b, "view", t)] = kept
    FM.clear(); VM.clear(); gc.collect()
_bad = [(b, k, t) for b in BACKENDS for k in ("frozen", "view") for t in TYPES
        if CAL29[b][k][t]["n"] < CAL_MIN or CAL29[b][k][t]["sigma"] is None or CAL29[b][k][t]["sigma"] <= 1e-9]
if _bad:
    raise SystemExit(f"stop: a spread of form ZP cannot be formed for a candidate ({_bad[0]})")
print("the spreads under each scoring (σ_F, σ_P per type):",
      {b: {t: (round(CAL29[b]["frozen"][t]["sigma"], 4), round(CAL29[b]["view"][t]["sigma"], 4)) for t in TYPES} for b in BACKENDS}, f"· {time.time() - t_c:.0f} s")

S29 = {}                                                              # (name, type, set, section) → scores of the section's test clips


def score29(name, t, set_, q):
    # a candidate ("K_DN", "F_KM", ...), a view part ("V_DN", ...) or a system as it is ("ZP", "frozen", "view_paper")
    k = (name, t, set_, q)
    if k not in S29:
        ref, te = dc_rows(t, set_, "train", q), dc_rows(t, set_, "test", q)
        if name in ("ZP", "frozen", "view_paper"):
            S29[k] = score_of(name, t, set_, q, SEEDS)
        else:
            kind, b = name.split("_")
            if kind == "F":
                S29[k] = frozen_b(b, t, ref, te)
            elif kind == "V":
                S29[k] = view_b(b, t, ref, te)
            else:
                S29[k] = (np.log(np.maximum(score29(f"F_{b}", t, set_, q), LDN_FLOOR)) / CAL29[b]["frozen"][t]["sigma"]
                          + np.log(np.maximum(score29(f"V_{b}", t, set_, q), LDN_FLOOR)) / CAL29[b]["view"][t]["sigma"])
    return S29[k]


NAMES29 = ["ZP", "frozen", "view_paper"] + CANDS + [f"V_{b}" for b in BACKENDS]
LABEL29 = {"ZP": "as it is · kept system (notebook 28)", "frozen": "as it is · frozen recipe alone", "view_paper": "as it is · view alone",
           **{f"K_{b}": f"K_{b} · frozen + view, both scored by {b}" for b in BACKENDS}, **{f"F_{b}": f"F_{b} · frozen recipe scored by {b}" for b in BACKENDS},
           **{f"V_{b}": f"V_{b} · view scored by {b} (a part, not a candidate)" for b in BACKENDS}}
DEV29, DEV29_SC = {nm: {} for nm in NAMES29}, {}
for t in TYPES:
    m = DMETA[t]
    for q in machines(t, "dev"):
        te = dc_rows(t, "dev", "test", q); y = np.array(m["label"])[te]; dom = DOM[t][te]
        DEV29_SC[(t, q)] = {"file": [m["name"][i] for i in te], "label": y.tolist(), "domain": dom.tolist()}
        for nm in NAMES29:
            s = score29(nm, t, "dev", q); assert np.isfinite(s).all(), (nm, t, q)
            DEV29[nm].setdefault(t, {})[q] = section_metrics(y, s, dom); DEV29_SC[(t, q)][nm] = s.tolist()
    FM.clear(); VM.clear(); gc.collect()
DEV29 = {nm: summary(DEV29[nm]) for nm in NAMES29}
print(f"\nDevelopment set · the official score Ω, the source, target and pAUC lines, and each type's score · {time.time() - t_c:.0f} s")
print(f"  {'':52s} {'Ω':>7s} {'source':>7s} {'target':>7s} {'pAUC':>7s}   " + " ".join(f"{NAME[t][:8]:>8s}" for t in TYPES))
for nm in NAMES29:
    d = DEV29[nm]
    print(f"  {LABEL29[nm]:52s} {d['omega']:7.4f} {d['AUC_source']:7.4f} {d['AUC_target']:7.4f} {d['pAUC']:7.4f}   " + " ".join(f"{d['per_type'][t]['score']:8.4f}" for t in TYPES))
# the choice, by the rules: the best candidate (the first in the rules' order on a tie), adopted only if it is at least ADOPT_BAR above
# the system as it is, both rounded to four decimals
_asis = round(DEV29["ZP"]["omega"], 4)
_best = max(CANDS, key=lambda c: (round(DEV29[c]["omega"], 4), -CANDS.index(c)))
CHOICE = {"as_it_is": _asis, "candidates": {c: round(DEV29[c]["omega"], 4) for c in CANDS}, "best": _best, "best_omega": round(DEV29[_best]["omega"], 4),
          "best_less_as_it_is": round(round(DEV29[_best]["omega"], 4) - _asis, 4), "bar": ADOPT_BAR}
CHOICE["adopted"] = bool(CHOICE["best_less_as_it_is"] >= ADOPT_BAR)
print(f"\nThe choice · the best candidate on the development set is {_best} ({CHOICE['best_omega']:.4f}); the system as it is has {_asis:.4f}; "
      f"the difference is {CHOICE['best_less_as_it_is']:+.4f} (adopted from {ADOPT_BAR:+.2f}) →",
      f"ADOPTED: {_best}" if CHOICE["adopted"] else "NOTHING IS ADOPTED; the evaluation set is not touched and notebook 28's result is the final one")

In [ ]:
# 7 · The evaluation set, once, and only for an adopted candidate. Its scores (and its parts') are written before the ground truth is
# fetched; then the metrics and the official evaluator, as in notebook 28. A later run reads the saved results and scores nothing again.
EV_DIR = OUT29 / "dcase2022_evaluation_n29"
EV_RES, EV_LOCK = EV_DIR / "results.json", EV_DIR / "scores_written.json"
N28 = json.loads((OUT / "domain_shift.json").read_text())             # notebook 28's result: the reference beside which this one is read
N28_FINAL = N28["final"]["numbers"]["kept"]
EVAL, OFFICIAL, FINAL, FINAL_SOURCE, Q1, EV_SETS = None, {}, {}, {}, None, {}
if EV_LOCK.exists():                                                  # the evaluation set was scored by an earlier run: that choice stands
    _lk = json.loads(EV_LOCK.read_text())
    if not CHOICE["adopted"] or _lk["chosen"] != CHOICE["best"]:
        raise SystemExit(f"stop: the evaluation set was already scored for {_lk['chosen']}, and this run chooses otherwise")
if CHOICE["adopted"]:
    _b = CHOICE["best"].split("_")[1]
    EV_SETS = {"chosen": CHOICE["best"]} if CHOICE["best"].startswith("F_") else {"chosen": CHOICE["best"], "frozen_part": f"F_{_b}", "view_part": f"V_{_b}"}
    EV_DIR.mkdir(exist_ok=True)
    GT_PARTS = ("ground_truth_data", "ground_truth_domain")               # the repository's label files and domain files
    EV_DIR.mkdir(exist_ok=True)
    clip_number = lambda name: int(re.search(r"(\d{4})\.wav$", name).group(1))


    def fetch_evaluator():
        # the official evaluator's repository at the pinned commit: the script and the ground-truth files of the evaluation set, checked
        # against the recorded hashes before anything is used
        d = WORK / "evaluator2022"
        if not (d / "checked").exists():
            z = WORK / EV_ARCHIVE[1]
            download(EV_ARCHIVE[0], z)
            with zipfile.ZipFile(z) as zf:
                names = {n.split("/", 1)[1]: n for n in zf.namelist() if "/" in n and not n.endswith("/")}
                want = sorted(n for n in names if n.startswith(tuple(p + "/" for p in GT_PARTS)) and n.endswith(".csv"))
                if EV_SCRIPT not in names or not want:
                    raise SystemExit("stop: the evaluator's repository does not hold the script and the ground truth")
                blobs = {n: zf.read(names[n]) for n in [EV_SCRIPT] + want}
            h = hashlib.sha256()
            for n in want:
                h.update(n.encode() + b"\0" + blobs[n])
            if hashlib.sha256(blobs[EV_SCRIPT]).hexdigest() != EV_SHA or h.hexdigest() != GT_SHA:
                raise SystemExit("stop: the evaluator's repository is not the recorded one (the script or a ground-truth file differs)")
            shutil.rmtree(d, ignore_errors=True)
            for n, b in blobs.items():
                (d / n).parent.mkdir(parents=True, exist_ok=True); (d / n).write_bytes(b)
            (d / "checked").write_text(f"{len(want)} ground-truth files")
        return d


    def read_ground_truth():
        # the labels and the domains of the evaluation test clips: (type, section) → clip number → (label, domain). The repository names a
        # clip section_03_0000.wav and the zips name it section_03_test_0000.wav; the number is what joins them
        d = fetch_evaluator(); gt = {}
        for t in TYPES:
            for q in EVAL_IDS[t]:
                with open(d / GT_PARTS[0] / f"ground_truth_{t}_{q}_test.csv", newline="") as fh:
                    lab = [r for r in csv.reader(fh) if r]
                with open(d / GT_PARTS[1] / f"ground_truth_{t}_{q}_test.csv", newline="") as fh:
                    dom = [r for r in csv.reader(fh) if r]
                assert [r[0] for r in lab] == [r[0] for r in dom] and len(set(clip_number(r[0]) for r in lab)) == len(lab), (t, q)
                gt[(t, q)] = {clip_number(a[0]): (int(float(a[1])), "target" if int(float(b[1])) == 1 else "source") for a, b in zip(lab, dom)}
        return d, gt


    def read_result(path):
        # one result file of the official evaluator: per type and section its AUC (source), AUC (target) and pAUC, and the lines over all
        res, cur = {"per_section": {}}, None
        with open(path, newline="") as fh:
            for row in csv.reader(fh):
                if len(row) == 1 and row[0]:
                    cur = row[0]
                elif len(row) >= 5 and re.fullmatch(r"\d\d", row[0]) and cur:
                    res["per_section"].setdefault(cur, {})["section_" + row[0]] = {"AUC_source": float(row[2]), "AUC_target": float(row[3]), "pAUC": float(row[4]), "AUC_all": float(row[1])}
                elif row and row[0] == "official score":
                    res["omega"] = float(row[2])
                elif row and row[0].startswith("source harmonic mean over all"):
                    res["AUC_source"] = float(row[2])
                elif row and row[0].startswith("target harmonic mean over all"):
                    res["AUC_target"] = float(row[2])
        return res


    def official_check():
        # the official evaluator (the pinned file, unmodified) on the three sets of score files, compared with this notebook's numbers
        out = {k: {"ran": False} for k in EV_SETS}
        info = {"archive": EV_ARCHIVE[0], "commit": EV_COMMIT, "script_sha256": EV_SHA, "ground_truth_sha256": GT_SHA}

        def stop(reason):
            return {k: {"ran": False, "reason": reason, "evaluator": info} for k in EV_SETS}
        lock = json.loads(EV_LOCK.read_text())
        if not EV_RES.exists() or any(not (EV_DIR / k / f).exists() for k in EV_SETS for f in lock["files"][k] + lock["decision_files"][k]):
            return stop("a score file or results.json of this notebook is missing")
        try:
            d = fetch_evaluator()
        except (SystemExit, Exception) as e:
            return stop(f"the evaluator's repository did not download, or it is not the recorded one ({str(e)[:160]})")
        wd = WORK / "official_evaluator_2022_n29"
        shutil.rmtree(wd, ignore_errors=True); wd.mkdir(parents=True)
        for p in GT_PARTS:
            shutil.copytree(d / p, wd / p)
        shutil.copy(d / EV_SCRIPT, wd / EV_SCRIPT)
        for k in EV_SETS:                                                 # the evaluator reads ./teams/<team>/<system>/
            (wd / "teams" / "P04" / k).mkdir(parents=True)
            for f in lock["files"][k] + lock["decision_files"][k]:
                shutil.copy(EV_DIR / k / f, wd / "teams" / "P04" / k / f)
        r = subprocess.run([sys.executable, EV_SCRIPT], cwd=wd, capture_output=True, text=True)   # the official script, unmodified
        if r.returncode != 0:
            return stop(f"the evaluator stopped (code {r.returncode}): {(r.stderr or r.stdout)[-300:]}")
        ours = json.loads(EV_RES.read_text())["systems"]
        for k in EV_SETS:
            res_f = wd / "teams_result" / f"{k}_result.csv"
            if not res_f.exists():
                out[k] = {"ran": False, "reason": "the evaluator wrote no result file for this set", "evaluator": info}
                continue
            off = read_result(res_f); mine = ours[k]
            if set(off["per_section"]) != set(TYPES) or any(set(off["per_section"][t]) != set(mine["per_type"][t]["per_section"]) for t in TYPES) or "omega" not in off:
                out[k] = {"ran": False, "reason": "the evaluator's result file does not list the same machine types and sections", "evaluator": info}
                continue
            diffs = [abs(off["per_section"][t][q][v] - mine["per_type"][t]["per_section"][q][v]) for t in TYPES for q in off["per_section"][t] for v in VALUES + ("AUC_all",)]
            diffs += [abs(off["omega"] - mine["omega"]), abs(off["AUC_source"] - mine["AUC_source"]), abs(off["AUC_target"] - mine["AUC_target"])]
            out[k] = {"ran": True, "agree": bool(max(diffs) <= EVAL_TOL), "max_abs_difference": float(max(diffs)), "values_compared": len(diffs), "evaluator": info,
                      "omega": off["omega"], "AUC_source": off["AUC_source"], "AUC_target": off["AUC_target"], "per_section": off["per_section"], "this_notebook_omega": mine["omega"]}
            shutil.copy(res_f, EV_DIR / k / "official_evaluator_result.csv")
        return out


    if EV_RES.exists():
        EVAL = json.loads(EV_RES.read_text())
        print(f"the evaluation set was scored on {EVAL['scores_written']}; its saved results are read, and nothing is scored again")
    else:
        if not EV_LOCK.exists():
            files, decisions = {k: [] for k in EV_SETS}, {k: [] for k in EV_SETS}
            for k in EV_SETS:
                (EV_DIR / k).mkdir(exist_ok=True)
            for t in TYPES:
                for q in machines(t, "eval"):
                    te = dc_rows(t, "eval", "test", q); names = [DMETA[t]["name"][i] for i in te]
                    for k, nm in EV_SETS.items():
                        sc = score29(nm, t, "eval", q)
                        assert len(sc) == len(te) and np.isfinite(sc).all(), (k, t, q)
                        f, g = EV_DIR / k / f"anomaly_score_{t}_{q}_test.csv", EV_DIR / k / f"decision_result_{t}_{q}_test.csv"
                        f.write_text("".join(f"{a},{float(b)!r}\n" for a, b in sorted(zip(names, sc))))
                        g.write_text("".join(f"{a},0\n" for a in sorted(names)))  # the evaluator needs a decision file; no threshold is set
                        files[k].append(f.name); decisions[k].append(g.name)
                FM.clear(); VM.clear()
            EV_LOCK.write_text(json.dumps({"written": time.strftime("%Y-%m-%d %H:%M"), "files": files, "decision_files": decisions, "test_run": TEST, "chosen": CHOICE["best"],
                                           "sets": EV_SETS, "choice": CHOICE, "calibration": CAL29[_b]}))
            print(f"the evaluation scores of {CHOICE['best']} and of its parts are written ({sum(len(v) for v in files.values())} files), "
                  "before any label is read", flush=True)
        lock = json.loads(EV_LOCK.read_text())
        _d, GT = read_ground_truth()                                  # the labels and the domains: fetched and read only here
        for p in GT_PARTS:
            (EV_DIR / p).mkdir(exist_ok=True)
            for t in TYPES:
                for q in EVAL_IDS[t]:
                    shutil.copy(_d / p / f"ground_truth_{t}_{q}_test.csv", EV_DIR / p / f"ground_truth_{t}_{q}_test.csv")
        _sys = {}
        for k in EV_SETS:
            per = {}
            for fn in lock["files"][k]:
                t, q = re.match(r"anomaly_score_(\w+?)_(section_\d+)_test\.csv$", fn).groups()
                rows = [ln.split(",") for ln in (EV_DIR / k / fn).read_text().splitlines() if ln]
                num = [clip_number(a) for a, _ in rows]
                assert sorted(num) == sorted(GT[(t, q)]), f"{fn}: the clips are not the ground truth's"
                per.setdefault(t, {})[q] = section_metrics([GT[(t, q)][n][0] for n in num], [float(b) for _, b in rows], [GT[(t, q)][n][1] for n in num])
            _sys[k] = {"system": EV_SETS[k], **summary({t: per[t] for t in TYPES})}
        EVAL = {"scored": True, "scores_written": lock["written"], "results_written": time.strftime("%Y-%m-%d %H:%M"), "chosen": CHOICE["best"], "test_run": TEST,
                "ground_truth": {"commit": EV_COMMIT, "sha256": GT_SHA}, "systems": _sys}
        EV_RES.write_text(json.dumps(EVAL, indent=1))
        del GT
    try:
        OFFICIAL = official_check()
    except Exception as e:                                            # this notebook's numbers do not depend on this check
        OFFICIAL = {k: {"ran": False, "reason": f"an unexpected error ({repr(e)[:200]})"} for k in EV_SETS}
    (EV_DIR / "official_evaluator_check.json").write_text(json.dumps(OFFICIAL, indent=1))


    def final_numbers(k, official=None):
        # the rules: if the official evaluator's numbers differ from this notebook's, the evaluator's are the result
        o = OFFICIAL[k] if official is None else official
        if o.get("ran") and not o.get("agree"):
            return summary({t: o["per_section"][t] for t in TYPES}), "the official evaluator (its numbers differ from this notebook's)"
        return ({kk: v for kk, v in EVAL["systems"][k].items() if kk != "system"},
                "this notebook (the official evaluator gives the same numbers)" if o.get("ran") else "this notebook (the official evaluator was not run)")


    def q1(omega):
        # the rules' word for the official score, in % of two decimals as the results page prints it, and its place among the 31 teams
        a = round(100 * omega, 2)
        word = ("above the first team" if a > Q1_FIRST else "with the leading teams" if a >= Q1_LEADING else "in the upper half" if a >= Q1_UPPER
                else "above the baselines" if a >= Q1_BASE else "below the baselines")
        above = [k_ for k_, v in TEAMS.items() if round(v, 2) > a]
        return {"omega_percent": a, "word": word, "place_among_the_31_teams": 1 + len(above), "teams_above": len(above), "next_team_above": above[-1] if above else None,
                "baselines_percent": {k_: round(v, 2) for k_, v in BASELINES.items()}, "first_team_percent": Q1_FIRST}


    for k in EV_SETS:
        FINAL[k], FINAL_SOURCE[k] = final_numbers(k)
        o = OFFICIAL[k]
        print(f"the official evaluator on the {k} scores ({EV_SETS[k]}): " + (f"the same numbers: {o['agree']} (largest difference {o['max_abs_difference']:.1e} over "
              f"{o['values_compared']} values; its official score {o['omega']:.6f})" if o["ran"] else f"NOT run: {o['reason']}"))
    Q1 = q1(FINAL["chosen"]["omega"]); _c = FINAL["chosen"]
    print(f"\nEvaluation set, scored once for {CHOICE['best']} ({EVAL['scores_written']}):")
    print(f"  {'':40s} {'Ω':>7s} {'source':>7s} {'target':>7s} {'pAUC':>7s}   " + " ".join(f"{NAME[t][:8]:>8s}" for t in TYPES))
    for lab, d in [("notebook 28 · the kept system as it is", N28_FINAL)] + [(f"{k} · {EV_SETS[k]}", FINAL[k]) for k in EV_SETS]:
        print(f"  {lab:40s} {d['omega']:7.4f} {d['AUC_source']:7.4f} {d['AUC_target']:7.4f} {d['pAUC']:7.4f}   " + " ".join(f"{d['per_type'][t]['score']:8.4f}" for t in TYPES))
    print(f"\n{CHOICE['best']}: official score {Q1['omega_percent']:.2f} % → {Q1['word'].upper()} · place {Q1['place_among_the_31_teams']} beside the 31 teams · "
          f"notebook 28's system as it is: {round(100 * N28_FINAL['omega'], 2):.2f} % · the difference: {100 * (_c['omega'] - N28_FINAL['omega']):+.2f} points")
else:
    print("nothing is adopted: the evaluation set is not touched. Notebook 28's result stands:", f"{round(100 * N28_FINAL['omega'], 2):.2f} %")

In [ ]:
# 8 · Save, one picture, and one zip to download
import matplotlib.pyplot as plt

RESULT = {"rules": "results/mimii_beats/RULES_domain_backends_2022_written_before_running.md", "date": time.strftime("%Y-%m-%d %H:%M"), "test_run": TEST,
          "meta": {"device": DEVICE, "types": TYPES, "development_sections": DEV_IDS, "evaluation_sections": EVAL_IDS, "seeds": SEEDS, "frozen": [[e, c, sd] for e, c, sd in FROZEN],
                   "frozen_cap": FROZEN_CAP, "km_centres": KM_CENTRES, "backends": BACKENDS, "candidates": CANDS, "prep_seconds": PREP_S,
                   "zips": {"development": {k: list(v[1:]) for k, v in DC_DEV.items()}, "additional": {k: list(v[1:]) for k, v in XZIPS.items()},
                            "evaluation": {k: list(v[1:]) for k, v in DC_EVAL.items()}},
                   "evaluator": {"archive": EV_ARCHIVE[0], "commit": EV_COMMIT, "script": EV_SCRIPT, "script_sha256": EV_SHA, "ground_truth_sha256": GT_SHA},
                   "notebook_28": {"date": N28["date"], "omega": N28_FINAL["omega"], "scores_written": N28["evaluation"]["scores_written"]},
                   "session_minutes": round((time.time() - T_START) / 60, 1)},
          "G0": G0, "G_FILES": GFILES, "G_SAME": GSAME, "counts": DCASE["counts"], "domains": DCASE["domains"],
          "calibration": {"minimum_clips": CAL_MIN, "frozen_cap": FROZEN_CAP, "backends": CAL29},
          "development": DEV29, "choice": CHOICE, "evaluation": EVAL, "official_evaluator": OFFICIAL,
          "final": {"numbers": FINAL, "source": FINAL_SOURCE, "sets": EV_SETS, "notebook_28_kept": N28_FINAL}, "question_1": Q1}
(OUT29 / "domain_backends.json").write_text(json.dumps(RESULT, indent=1, default=float, ensure_ascii=False))
SC_DIR = OUT29 / "development_scores"; SC_DIR.mkdir(exist_ok=True)
for (t, q), d in DEV29_SC.items():
    (SC_DIR / f"scores_{t}_{q}.csv").write_text("file,label,domain," + ",".join(NAMES29) + "\n" + "".join(
        f"{d['file'][i]},{int(d['label'][i])},{d['domain'][i]}," + ",".join(repr(float(d[nm][i])) for nm in NAMES29) + "\n" for i in range(len(d["file"]))))
CAL_DIR = OUT29 / "calibration_scores"; CAL_DIR.mkdir(exist_ok=True)
for (b, k, t), rows in CAL29_SC.items():
    (CAL_DIR / f"{k}_{b}_{t}.csv").write_text("file,score\n" + "".join(f"{a},{float(s)!r}\n" for a, s in rows))

INK, MUTED, GRID = "#0b0b0b", "#52514e", "#e4e3df"
C_REF, C_NEW = "#2a78d6", "#1baf7a"
fig, axes = plt.subplots(1, 2, figsize=(14.5, 6.2), gridspec_kw={"width_ratios": [1.0, 1.25]})
ax = axes[0]; order = ["ZP"] + CANDS; vals = [DEV29[nm]["omega"] for nm in order]
ax.barh(range(len(order)), vals, color=[C_REF] + [C_NEW if (CHOICE["adopted"] and nm == CHOICE["best"]) else "#b9b8b3" for nm in CANDS], height=0.62)
ax.axvline(DEV29["ZP"]["omega"] + ADOPT_BAR, color=INK, lw=1.0, ls=(0, (4, 2)))
ax.text(DEV29["ZP"]["omega"] + ADOPT_BAR, -0.75, " adopted from here", fontsize=8.5, color=INK, va="center")
for i, v in enumerate(vals):
    ax.text(v + 0.003, i, f"{v:.4f}", va="center", fontsize=8.5, color=INK)
ax.set_yticks(range(len(order)), ["as it is (notebook 28)"] + CANDS); ax.invert_yaxis()
ax.set_xlim(max(0.0, min(vals) - 0.08), max(vals) + 0.06); ax.set_ylim(len(order) - 0.4, -1.1)
ax.set_xlabel("official score on the development set; the scale does not start at zero", color=MUTED, fontsize=8.5)
ax.set_title("Development set: the six candidates" + (f" · adopted: {CHOICE['best']}" if CHOICE["adopted"] else " · nothing is adopted"), loc="left", fontsize=10.5, color=INK)
ax = axes[1]; x = np.arange(len(TYPES) + 1)
ref = [N28_FINAL["per_type"][t]["score"] for t in TYPES] + [N28_FINAL["omega"]]
ax.plot(x - 0.12, ref, "o", ms=8.5, color=C_REF, mec="white", mew=1.2, label="notebook 28: the kept system as it is", zorder=4)
if CHOICE["adopted"]:
    new = [FINAL["chosen"]["per_type"][t]["score"] for t in TYPES] + [FINAL["chosen"]["omega"]]
    ax.plot(x + 0.12, new, "D", ms=8.5, color=C_NEW, mec="white", mew=1.2, label=f"this notebook: {CHOICE['best']}", zorder=4)
    for i in range(len(x)):
        ax.plot([x[i] - 0.12, x[i] + 0.12], [ref[i], new[i]], color=MUTED, lw=1.0, zorder=2)
    ax.annotate(f"{new[-1]:.4f}", (x[-1] + 0.12, new[-1]), xytext=(0, 9), textcoords="offset points", ha="center", fontsize=8.5, color=INK)
ax.annotate(f"{ref[-1]:.4f}", (x[-1] - 0.12, ref[-1]), xytext=(0, -15), textcoords="offset points", ha="center", fontsize=8.5, color=INK)
ax.hlines([v / 100 for v in TEAMS.values()], x[-1] + 0.34, x[-1] + 0.46, color=INK, lw=1.1, zorder=3, label="the challenge's 31 teams (each team's best system)")
ax.axvline(x[-1] - 0.5, color=GRID, lw=1.0)
ax.set_xticks(x, [NAME[t] for t in TYPES] + ["all types\n(official score Ω)"]); ax.set_xlim(-0.6, len(TYPES) + 0.6)
_lo = max(0.0, min(0.4, math.floor(min(ref + (new if CHOICE["adopted"] else [])) * 20) / 20 - 0.05)); ax.set_ylim(_lo, 1.0)
ax.set_title("Evaluation set" + (f", scored once for {CHOICE['best']} ({EVAL['scores_written']})" if CHOICE["adopted"] else ": not touched by this notebook")
             + ("; the scale does not start at zero" if _lo > 0 else ""), loc="left", fontsize=10.5, color=INK)
ax.legend(frameon=False, fontsize=8.5, loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=2)
for ax in axes:
    ax.grid(axis="y" if ax is axes[1] else "x", color=GRID, lw=0.6); ax.set_axisbelow(True); ax.tick_params(colors=MUTED)
    for sp in ("top", "right"):
        ax.spines[sp].set_visible(False)
fig.suptitle("P04 · notebook 29 · a scoring that knows the two domains (DCASE 2022 Task 2, no training)\n"
             + (f"{CHOICE['best']}: official score {Q1['omega_percent']:.2f} % → {Q1['word']} · place {Q1['place_among_the_31_teams']} beside the 31 teams · "
                f"notebook 28: {round(100 * N28_FINAL['omega'], 2):.2f} %" if CHOICE["adopted"] else f"nothing is adopted · notebook 28's {round(100 * N28_FINAL['omega'], 2):.2f} % stands"),
             fontsize=11, x=0.012, y=0.985, ha="left")
fig.subplots_adjust(left=0.13, right=0.99, top=0.83, bottom=0.2, wspace=0.12); fig.savefig(OUT29 / "domain_backends.png", dpi=150); plt.show()

ZIP = OUT29 / "p04_n29_results.zip"
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for f in ("domain_backends.json", "domain_backends.png"):
        z.write(OUT29 / f, f)
    for d in (SC_DIR, CAL_DIR):
        for f in sorted(d.glob("*.csv")):
            z.write(f, f"{d.name}/{f.name}")
    if EV_DIR.exists():
        for f in sorted(p for p in EV_DIR.rglob("*") if p.is_file()):
            z.write(f, f"{EV_DIR.name}/{f.relative_to(EV_DIR).as_posix()}")
print(f"saved {ZIP} · {(time.time() - T_START) / 60:.0f} min in this session")
if IN_COLAB:
    from google.colab import files
    files.download(str(ZIP))

## Send back
Send the zip that downloaded (`p04_n29_results.zip`). It is also in Drive › `P04_DomainShift/n29_backends`.